# Non-DOI Harvest

Harvesting datasets metadata for responsitories not issuing DOIs

## Import

In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import os

from sindex.sources.emdb.jobs import (
    batch_slim_emdb_record_to_ndjson,
    harvest_emdb_datasets_for_date_range_to_ndjson,
)
from sindex.sources.emdb.normalize import slim_emdb_record

## Electron Microscopy Data Bank (EMDB)

### Get EMDB record using api

In [5]:
start_date_str = "2025-10-01"
end_date_str = "2026-04-30" 
out_folder = r"D:\may-2026-data\records\raw-records\emdb-records"

In [7]:
harvest_emdb_datasets_for_date_range_to_ndjson(start_date_str, end_date_str, out_folder)

Fetching EMDB IDs from CSV search endpoint...
Fetched 57595 EMDB IDs.
Harvesting entries from 2025-10-01 to 2026-04-30 using 12 workers...
Processed 57595/57595 IDs (written: 1557)

Done. Wrote 1557 entries to D:\may-2026-data\records\raw-records\emdb-records\emdb_deposited_2025-10-01_to_2026-04-30.ndjson


1557

### Create slim metadata record

In [8]:
emdb_record_path = (
    r"D:\pipeline-data\records\raw-records\emdb-records\emdb-records.ndjson"
)

#### Test with one record

In [9]:
with open(emdb_record_path, "r", encoding="utf-8") as f:
    first_line = f.readline()
    if first_line:
        first_record = json.loads(first_line)
        print("First record:", first_record)

First record: {'_id': '6743ad09c7dd9684974495f4', 'admin': {'authors_list': {'author': [{'instance_type': 'author', 'valueOf_': 'Markert J'}, {'instance_type': 'author', 'valueOf_': 'Farnung L'}]}, 'current_status': {'code': {'valueOf_': 'REL'}, 'date': '2025-11-26T00:00:00', 'processing_site': 'RCSB'}, 'grant_support': {'grant_reference': [{'country': 'United States', 'funding_body': 'National Institutes of Health/National Institute of Environmental Health Sciences (NIH/NIEHS)', 'instance_type': 'grant_reference'}, {'country': 'United States', 'funding_body': 'Richard and Susan Smith Family Foundation', 'instance_type': 'grant_reference'}, {'country': 'United States', 'funding_body': 'Damon Runyon Cancer Research Foundation', 'instance_type': 'grant_reference'}, {'country': 'United States', 'funding_body': 'Rita Allen Foundation', 'instance_type': 'grant_reference'}]}, 'key_dates': {'deposition': '2024-11-21T00:00:00', 'header_release': '2025-11-26T00:00:00', 'map_release': '2025-11-2

In [10]:
slim_record = slim_emdb_record(metadata=first_record)

In [11]:
display(slim_record)

{'source': 'emdb',
 'identifiers': [{'identifier': 'EMD-48024', 'identifier_type': 'emdb_id'}],
 'url': 'https://www.ebi.ac.uk/emdb/EMD-48024',
 'title': 'map beta Paf1C',
 'subjects': ['Transcription', 'SETD2', 'H3K36me3'],
 'publication_date': '2024-11-21T00:00:00',
 'pubyear': 2024,
 'creators': [{'name': 'Markert J', 'name_type': 'Personal'},
  {'name': 'Farnung L', 'name_type': 'Personal'}],
 'publisher': 'The Electron Microscopy Data Bank (EMDB)'}

### Batch slim record

In [1]:
raw_emdb_record_folder = r"D:\may-2026-data\records\raw-records\emdb-records"
slim_emdb_record_folder = r"D:\may-2026-data\records\slim-records\emdb-slim-records"

In [13]:
summary = batch_slim_emdb_record_to_ndjson(
    src_folder=raw_emdb_record_folder,
    dst_folder=slim_emdb_record_folder,
    overwrite=True,
    one_line_progress=True,
)

[1/1] files completed
Done. files=1 kept=1,557 bad=0 time=0.5s rate≈3,255/s → D:\may-2026-data\records\slim-records\emdb-slim-records


### Export as csv for Software Heritage processing

In [2]:
import json, csv, os

folder = slim_emdb_record_folder
output_file = r"D:\may-2026-data\records\ids\emdb_ids.csv"

with open(output_file, 'w', newline='') as out:
    writer = csv.writer(out)
    writer.writerow(['dataset_id'])
    for filename in os.listdir(folder):
        if filename.endswith('.ndjson'):
            with open(os.path.join(folder, filename)) as f:
                for line in f:
                    obj = json.loads(line)
                    writer.writerow([obj['identifiers'][0]['identifier']])